# Tree-of-Thought Linearizer

Convert Tree-of-Thought JSON trees into linearized training traces with controlled backtracking.

Supports three modes:
- `no_backtrack`: Clean solution path only
- `one_backtrack`: One failed attempt before solution
- `two_backtrack`: Two failed attempts before solution

In [1]:
import json
from pathlib import Path
from typing import Dict, List, Tuple, Optional

In [2]:
def find_successful_leaf(tree: Dict) -> Optional[Dict]:
    """
    Find a leaf node where the state equals [24] or [24.0].
    
    Args:
        tree: The complete ToT tree JSON
        
    Returns:
        A node dict representing a successful solution, or None
    """
    nodes = tree.get('nodes', [])
    
    # Find all solution nodes
    solution_nodes = [
        node for node in nodes 
        if node.get('is_solution', False)
    ]
    
    if not solution_nodes:
        return None
    
    # Return the first successful solution (deterministic)
    # Sort by node id for determinism
    solution_nodes.sort(key=lambda n: n.get('id', 0))
    return solution_nodes[0]

In [3]:
def extract_solution_path(tree: Dict, leaf_node: Dict) -> List[Dict]:
    """
    Extract the path from root to successful leaf by following parent_id.
    
    Args:
        tree: The complete ToT tree JSON
        leaf_node: The successful leaf node
        
    Returns:
        List of nodes from root to leaf (in that order)
    """
    # Create node lookup map
    node_map = {node['id']: node for node in tree.get('nodes', [])}
    
    # Backtrack from leaf to root
    path = []
    current_id = leaf_node['id']
    
    while current_id is not None:
        current_node = node_map.get(current_id)
        if current_node is None:
            break
        path.insert(0, current_node)  # Insert at beginning for root->leaf order
        current_id = current_node.get('parent_id')
    
    return path

In [4]:
def select_failed_siblings(tree: Dict, solution_path: List[Dict], max_backtracks: int) -> List[Tuple[int, Dict]]:
    """
    Select failed sibling branches to show backtracking.
    Only considers shallow depth (depth 1 preferred).
    
    Args:
        tree: The complete ToT tree JSON
        solution_path: The successful path from root to leaf
        max_backtracks: Maximum number of failed branches to include
        
    Returns:
        List of tuples: (insertion_point_index, failed_node)
        insertion_point_index indicates where in solution_path this sibling diverges
    """
    if max_backtracks == 0 or len(solution_path) < 2:
        return []
    
    # Create node lookup map
    node_map = {node['id']: node for node in tree.get('nodes', [])}
    
    failed_branches = []
    
    # Only consider the first correct step (depth 1) for backtracking
    # This keeps it shallow and focused
    correct_node = solution_path[1]  # First step after root
    parent_id = correct_node.get('parent_id')
    
    if parent_id is None:
        return []
    
    parent_node = node_map.get(parent_id)
    if parent_node is None:
        return []
    
    # Get all children of the parent (siblings of the correct node)
    sibling_ids = parent_node.get('children_ids', [])
    
    # Filter to get failed siblings (not in solution path)
    solution_ids = {node['id'] for node in solution_path}
    failed_sibling_ids = [sid for sid in sibling_ids if sid not in solution_ids]
    
    # Get the actual nodes and sort by id for determinism
    failed_siblings = [node_map[sid] for sid in failed_sibling_ids if sid in node_map]
    failed_siblings.sort(key=lambda n: n.get('id', 0))
    
    # Take up to max_backtracks siblings
    for sibling in failed_siblings[:max_backtracks]:
        # Insert at index 1 (after root, before first correct step)
        failed_branches.append((1, sibling))
    
    return failed_branches

In [5]:
def format_step(node: Dict) -> str:
    """
    Format a single step as: operation → [state]
    
    Args:
        node: A node from the tree
        
    Returns:
        Formatted string for the step
    """
    # Get the thought/operation description
    thought = node.get('codeact', {}).get('thought', '')
    
    # Get the resulting state
    state = node.get('state', '')
    
    # If thought is empty, try to use action
    if not thought:
        action = node.get('action', '')
        if action and action != 'START':
            thought = action
    
    # Clean up the thought - extract just the operation part
    if '→' in thought:
        # Extract the part before the arrow
        thought = thought.split('→')[0].strip()
    
    # Format the output
    if thought and state:
        return f"{thought} → {state}"
    elif state:
        return f"Operation → {state}"
    else:
        return "Operation performed"

In [6]:
def get_initial_numbers(tree: Dict) -> List[int]:
    """
    Extract initial numbers from the root node.
    
    Args:
        tree: The complete ToT tree JSON
        
    Returns:
        List of initial numbers
    """
    nodes = tree.get('nodes', [])
    root = next((n for n in nodes if n.get('id') == 1), None)
    
    if root:
        obs = root.get('codeact', {}).get('observation', '')
        if 'Starting numbers:' in obs:
            import re
            match = re.search(r'\[([\d,\s]+)\]', obs)
            if match:
                nums = [int(n.strip()) for n in match.group(1).split(',')]
                return nums
    
    return []

In [7]:
def build_final_expression(solution_path: List[Dict]) -> str:
    """
    Build the final mathematical expression from the solution path.
    
    Args:
        solution_path: Path from root to successful leaf
        
    Returns:
        Mathematical expression string
    """
    # Try to extract from path_history of the final node
    if solution_path:
        final_node = solution_path[-1]
        path_history = final_node.get('path_history', '')
        
        if path_history:
            # Extract expressions from path history
            lines = path_history.strip().split('\n')
            expressions = []
            
            for line in lines:
                # Look for lines with operations
                if '=' in line and any(op in line for op in ['+', '-', '*', '/']):
                    # Extract the operation
                    if ':' in line:
                        expr = line.split(':', 1)[0].strip()
                        expressions.append(expr)
            
            if expressions:
                return ' → '.join(expressions)
    
    # Fallback: build from individual steps
    expressions = []
    for i, node in enumerate(solution_path[1:], 1):  # Skip root
        thought = node.get('codeact', {}).get('thought', '')
        if thought:
            # Extract operation part
            if '[' in thought:
                op_part = thought.split('[')[0].strip()
                if any(op in op_part for op in ['+', '-', '*', '/']):
                    expressions.append(op_part)
    
    if expressions:
        return ' → '.join(expressions)
    
    return "Expression not available"

In [8]:
def convert_tree_to_trace(tree_json: Dict, mode: str = "no_backtrack") -> str:
    """
    Convert a Tree-of-Thought JSON tree into a linearized training trace.
    
    Args:
        tree_json: The complete ToT tree JSON structure
        mode: One of "no_backtrack", "one_backtrack", or "two_backtrack"
        
    Returns:
        Formatted trace string following the template
    """
    # Validate mode
    valid_modes = ["no_backtrack", "one_backtrack", "two_backtrack"]
    if mode not in valid_modes:
        raise ValueError(f"Mode must be one of {valid_modes}, got: {mode}")
    
    # Step 1: Find successful leaf
    leaf = find_successful_leaf(tree_json)
    if leaf is None:
        return "### Error: No successful solution found in tree\n"
    
    # Step 2: Extract solution path
    solution_path = extract_solution_path(tree_json, leaf)
    if len(solution_path) < 2:
        return "### Error: Solution path too short\n"
    
    # Step 3: Get initial numbers
    initial_nums = get_initial_numbers(tree_json)
    
    # Step 4: Determine backtracking
    max_backtracks = {"no_backtrack": 0, "one_backtrack": 1, "two_backtrack": 2}[mode]
    failed_branches = select_failed_siblings(tree_json, solution_path, max_backtracks)
    
    # Step 5: Build the formatted output
    output = []
    output.append("### Problem:")
    output.append(f"Numbers: {' '.join(map(str, initial_nums))}")
    output.append("Target: 24")
    output.append("")
    output.append("### Reasoning:")
    output.append("")
    
    # Add failed attempts if any
    attempt_num = 1
    for idx, failed_node in failed_branches:
        output.append(f"Attempt {attempt_num}:")
        output.append(format_step(failed_node))
        output.append("This path does not lead to 24.")
        output.append("")
        output.append("Backtrack.")
        output.append("")
        attempt_num += 1
    
    # Add successful path
    if failed_branches:
        output.append("Continue:")
    else:
        output.append("Solution:")
    output.append("")
    
    # Format each step in the solution path (skip root)
    for i, node in enumerate(solution_path[1:], 1):
        step_text = format_step(node)
        output.append(step_text)
    
    output.append("")
    output.append("### Final Expression:")
    
    # Try to build expression
    final_expr = build_final_expression(solution_path)
    output.append(f"{final_expr} = 24")
    
    return "\n".join(output)

## Testing

Let's test the linearizer on one of the solved problems.

In [9]:
# Load a sample JSON file
sample_file = r"g:\class codes\tree-of-thought-llm\tot tree open ai\game24_codeact_tree_20260203_182742.json"

with open(sample_file, 'r', encoding='utf-8') as f:
    tree_data = json.load(f)

print(f"Loaded tree with {len(tree_data['nodes'])} nodes")
print(f"Solutions found: {tree_data['metadata']['statistics']['solutions_found']}")

Loaded tree with 88 nodes
Solutions found: 2


In [10]:
# Test mode: no_backtrack
print("=" * 80)
print("MODE: no_backtrack")
print("=" * 80)
trace = convert_tree_to_trace(tree_data, mode="no_backtrack")
print(trace)

MODE: no_backtrack
### Problem:
Numbers: 1 2 4 7
Target: 24

### Reasoning:

Solution:

Add 1 and 7 to get 8 [1 + 7 = 8] → [8, 2, 4]
Subtract 2 from 8 to get 6 [8 - 2 = 6] → [6, 4]
Multiply 6 and 4 to get 24 [6 * 4 = 24] → [24]

### Final Expression:
Subtract 2 from 8 to get 6 [8 - 2 = 6] → left = 24


In [11]:
# Test mode: one_backtrack
print("\n" + "=" * 80)
print("MODE: one_backtrack")
print("=" * 80)
trace = convert_tree_to_trace(tree_data, mode="one_backtrack")
print(trace)


MODE: one_backtrack
### Problem:
Numbers: 1 2 4 7
Target: 24

### Reasoning:

Attempt 1:
Subtract 1 from 4 to get 3 [4 - 1 = 3] → [3, 2, 7]
This path does not lead to 24.

Backtrack.

Continue:

Add 1 and 7 to get 8 [1 + 7 = 8] → [8, 2, 4]
Subtract 2 from 8 to get 6 [8 - 2 = 6] → [6, 4]
Multiply 6 and 4 to get 24 [6 * 4 = 24] → [24]

### Final Expression:
Subtract 2 from 8 to get 6 [8 - 2 = 6] → left = 24


In [12]:
# Test mode: two_backtrack
print("\n" + "=" * 80)
print("MODE: two_backtrack")
print("=" * 80)
trace = convert_tree_to_trace(tree_data, mode="two_backtrack")
print(trace)


MODE: two_backtrack
### Problem:
Numbers: 1 2 4 7
Target: 24

### Reasoning:

Attempt 1:
Subtract 1 from 4 to get 3 [4 - 1 = 3] → [3, 2, 7]
This path does not lead to 24.

Backtrack.

Attempt 2:
Multiply 2 and 4 to get 8 [2 * 4 = 8] → [8, 1, 7]
This path does not lead to 24.

Backtrack.

Continue:

Add 1 and 7 to get 8 [1 + 7 = 8] → [8, 2, 4]
Subtract 2 from 8 to get 6 [8 - 2 = 6] → [6, 4]
Multiply 6 and 4 to get 24 [6 * 4 = 24] → [24]

### Final Expression:
Subtract 2 from 8 to get 6 [8 - 2 = 6] → left = 24


## Batch Processing

Process all solved problems and generate training traces.

In [ ]:
def process_all_trees(folder_path: str, mode: str = "no_backtrack", output_file: str = None):
    """
    Process all JSON tree files in a folder and generate training traces.
    
    Args:
        folder_path: Path to folder containing JSON files
        mode: Backtracking mode
        output_file: Optional output file to save all traces
        
    Returns:
        List of (filename, trace) tuples
    """
    folder = Path(folder_path)
    json_files = sorted(folder.glob('game24_codeact_tree_*.json'))
    
    results = []
    successful = 0
    failed = 0
    
    for json_file in json_files:
        try:
            with open(json_file, 'r', encoding='utf-8') as f:
                tree_data = json.load(f)
            
            # Check if tree has solution
            if tree_data.get('metadata', {}).get('statistics', {}).get('solutions_found', 0) > 0:
                trace = convert_tree_to_trace(tree_data, mode=mode)
                results.append((json_file.name, trace))
                successful += 1
            else:
                failed += 1
        except Exception as e:
            print(f"Error processing {json_file.name}: {e}")
            failed += 1
    
    # Save to file if requested
    if output_file and results:
        with open(output_file, 'w', encoding='utf-8') as f:
            for filename, trace in results:
                f.write(f"# Source: {filename}\n")
                f.write(trace)
                f.write("\n\n" + "="*100 + "\n\n")
        print(f"\nSaved {len(results)} traces to {output_file}")
    
    print(f"\nProcessed {len(json_files)} files:")
    print(f"  ✓ Successful: {successful}")
    print(f"  ✗ Failed/No solution: {failed}")
    
    return results

In [ ]:
# Generate training data for all three modes
folder_path = r"g:\class codes\tree-of-thought-llm\tot tree open ai"

print("Generating training traces for all modes...\n")

# Mode 1: No backtrack
print("\n" + "="*80)
print("Processing: no_backtrack mode")
print("="*80)
results_no_bt = process_all_trees(
    folder_path, 
    mode="no_backtrack",
    output_file=r"g:\class codes\tree-of-thought-llm\tot tree open ai\training_traces_no_backtrack.txt"
)

# Mode 2: One backtrack
print("\n" + "="*80)
print("Processing: one_backtrack mode")
print("="*80)
results_one_bt = process_all_trees(
    folder_path, 
    mode="one_backtrack",
    output_file=r"g:\class codes\tree-of-thought-llm\tot tree open ai\training_traces_one_backtrack.txt"
)

# Mode 3: Two backtrack
print("\n" + "="*80)
print("Processing: two_backtrack mode")
print("="*80)
results_two_bt = process_all_trees(
    folder_path, 
    mode="two_backtrack",
    output_file=r"g:\class codes\tree-of-thought-llm\tot tree open ai\training_traces_two_backtrack.txt"
)

print("\n" + "="*80)
print("All training traces generated successfully!")
print("="*80)